# Match Outcome Prediction in Cricket and Football
### Using Random Forest and Artificial Neural Networks

This notebook implements the project described in the abstract:
*"Match Outcome Prediction in Cricket and Football Using Random Forest and Artificial Neural Networks"*.

**What this notebook does:**
1. Loads a football dataset (English Premier League, `E0 2018-2019.csv`) and a cricket dataset (IPL ball-by-ball data, `deliveries.csv`).
2. Engineers match-level features for both sports.
3. Trains a **Random Forest** classifier and an **Artificial Neural Network (ANN)** for each sport.
4. Evaluates and compares both models using **Accuracy, Precision, Recall, and F1-score**.
5. Visualizes feature importance and model comparison.

> **How to use in Google Colab:** Upload `E0 2018-2019.csv` and `deliveries.csv` via the Files panel first, then run the cells top to bottom. No other setup is required — all libraries used are pre-installed in Colab.

## 0. Setup: Import Libraries

In [ ]:
# Core libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Machine learning
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, classification_report, confusion_matrix)

# Deep learning
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Reproducibility
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (7, 4)

print("TensorFlow version:", tf.__version__)
print("All libraries imported successfully.")

## 1. Load the Datasets

Upload the two files yourself via the Colab **Files** panel (left sidebar → upload icon) before running the cell below:
- `E0 2018-2019.csv` (football match data)
- `deliveries.csv` (cricket ball-by-ball data)


In [ ]:
import os

FOOTBALL_FILE = "E0 2018-2019.csv"  # Changed from "E0_2018-2019.csv"
CRICKET_FILE = "deliveries.csv"

assert os.path.exists(FOOTBALL_FILE), f"{FOOTBALL_FILE} not found — please upload it via the Files panel."
assert os.path.exists(CRICKET_FILE), f"{CRICKET_FILE} not found — please upload it via the Files panel."
print("Both datasets are ready.")


---
# PART 1 — Football Match Outcome Prediction

**Goal:** Predict the Full-Time Result (`FTR`): Home win (H), Draw (D), or Away win (A), using match statistics and pre-match bookmaker odds.

### 1.1 Load and Explore the Data

In [ ]:
football_df = pd.read_csv(FOOTBALL_FILE)
print("Shape:", football_df.shape)
football_df.head()

In [ ]:
print("Missing values in key columns:")
print(football_df[['FTHG','FTAG','FTR','HS','AS','HST','AST']].isna().sum())

print("\nClass distribution (Full-Time Result):")
print(football_df['FTR'].value_counts())

sns.countplot(x='FTR', data=football_df, order=['H','D','A'], palette='viridis')
plt.title("Distribution of Match Outcomes (Football)")
plt.xlabel("Result (H=Home win, D=Draw, A=Away win)")
plt.ylabel("Number of matches")
plt.show()

### 1.2 Feature Engineering

We use in-match statistics (shots, shots on target, fouls, corners, cards) together with pre-match bookmaker odds (Bet365: `B365H`, `B365D`, `B365A`) as predictive features. The target is `FTR` (Full-Time Result).

In [ ]:
football_features = ['HS','AS','HST','AST','HF','AF','HC','AC','HY','AY','HR','AR',
                      'B365H','B365D','B365A']

football_df = football_df.dropna(subset=football_features + ['FTR']).reset_index(drop=True)

X_fb = football_df[football_features].copy()

le_fb = LabelEncoder()
y_fb = le_fb.fit_transform(football_df['FTR'])   # A=0, D=1, H=2 (alphabetical)
print("Classes:", list(le_fb.classes_))

X_fb.head()

In [ ]:
X_fb_train, X_fb_test, y_fb_train, y_fb_test = train_test_split(
    X_fb, y_fb, test_size=0.2, random_state=SEED, stratify=y_fb
)

scaler_fb = StandardScaler()
X_fb_train_s = scaler_fb.fit_transform(X_fb_train)
X_fb_test_s = scaler_fb.transform(X_fb_test)

print("Train size:", X_fb_train.shape, " Test size:", X_fb_test.shape)

### 1.3 Random Forest Model (Football)

In [ ]:
rf_fb = RandomForestClassifier(n_estimators=300, max_depth=None, random_state=SEED)
rf_fb.fit(X_fb_train, y_fb_train)

rf_fb_pred = rf_fb.predict(X_fb_test)

rf_fb_acc = accuracy_score(y_fb_test, rf_fb_pred)
rf_fb_prec = precision_score(y_fb_test, rf_fb_pred, average='macro', zero_division=0)
rf_fb_rec = recall_score(y_fb_test, rf_fb_pred, average='macro', zero_division=0)
rf_fb_f1 = f1_score(y_fb_test, rf_fb_pred, average='macro', zero_division=0)

print("Random Forest — Football")
print(f"Accuracy : {rf_fb_acc:.4f}")
print(f"Precision: {rf_fb_prec:.4f}")
print(f"Recall   : {rf_fb_rec:.4f}")
print(f"F1-score : {rf_fb_f1:.4f}")
print("\n", classification_report(y_fb_test, rf_fb_pred, target_names=le_fb.classes_, zero_division=0))

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_fb_test, rf_fb_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=le_fb.classes_, yticklabels=le_fb.classes_)
plt.title("Random Forest Confusion Matrix — Football")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

# Feature importance
importances = pd.Series(rf_fb.feature_importances_, index=football_features).sort_values(ascending=True)
importances.plot(kind='barh', color='teal')
plt.title("Random Forest Feature Importance — Football")
plt.xlabel("Importance")
plt.show()

### 1.4 Artificial Neural Network (ANN) Model (Football)

In [ ]:
def build_ann(input_dim, num_classes):
    model = keras.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(64, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(32, activation='relu'),
        layers.Dropout(0.2),
        layers.Dense(num_classes, activation='softmax')
    ])
    model.compile(optimizer='adam',
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])
    return model

ann_fb = build_ann(X_fb_train_s.shape[1], num_classes=len(le_fb.classes_))
ann_fb.summary()

In [ ]:
early_stop = keras.callbacks.EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)

history_fb = ann_fb.fit(
    X_fb_train_s, y_fb_train,
    validation_split=0.15,
    epochs=150,
    batch_size=16,
    callbacks=[early_stop],
    verbose=0
)

print("Training complete. Epochs run:", len(history_fb.history['loss']))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(history_fb.history['loss'], label='Train Loss')
ax[0].plot(history_fb.history['val_loss'], label='Val Loss')
ax[0].set_title("ANN Loss — Football")
ax[0].set_xlabel("Epoch"); ax[0].legend()

ax[1].plot(history_fb.history['accuracy'], label='Train Acc')
ax[1].plot(history_fb.history['val_accuracy'], label='Val Acc')
ax[1].set_title("ANN Accuracy — Football")
ax[1].set_xlabel("Epoch"); ax[1].legend()
plt.tight_layout()
plt.show()

In [ ]:
ann_fb_probs = ann_fb.predict(X_fb_test_s, verbose=0)
ann_fb_pred = np.argmax(ann_fb_probs, axis=1)

ann_fb_acc = accuracy_score(y_fb_test, ann_fb_pred)
ann_fb_prec = precision_score(y_fb_test, ann_fb_pred, average='macro', zero_division=0)
ann_fb_rec = recall_score(y_fb_test, ann_fb_pred, average='macro', zero_division=0)
ann_fb_f1 = f1_score(y_fb_test, ann_fb_pred, average='macro', zero_division=0)

print("ANN — Football")
print(f"Accuracy : {ann_fb_acc:.4f}")
print(f"Precision: {ann_fb_prec:.4f}")
print(f"Recall   : {ann_fb_rec:.4f}")
print(f"F1-score : {ann_fb_f1:.4f}")
print("\n", classification_report(y_fb_test, ann_fb_pred, target_names=le_fb.classes_, zero_division=0))

### 1.5 Football: Random Forest vs ANN Comparison

In [ ]:
football_results = pd.DataFrame({
    'Model': ['Random Forest', 'ANN'],
    'Accuracy': [rf_fb_acc, ann_fb_acc],
    'Precision': [rf_fb_prec, ann_fb_prec],
    'Recall': [rf_fb_rec, ann_fb_rec],
    'F1-score': [rf_fb_f1, ann_fb_f1]
})
display(football_results)

football_results.set_index('Model').plot(kind='bar', figsize=(8,5), colormap='Set2')
plt.title("Football: Random Forest vs ANN")
plt.ylabel("Score")
plt.ylim(0, 1)
plt.xticks(rotation=0)
plt.legend(loc='lower right')
plt.show()

### 1.6 Predict a Future Football Match

Both models are now trained, so we can use them to predict a **new, upcoming fixture** between two teams.

Exact in-match numbers (shots, corners, cards, etc.) obviously aren't known before kickoff, so we estimate them from each team's **historical averages** in the dataset — the home team's typical numbers when playing at home, and the away team's typical numbers when playing away. You just supply the two team names (and optionally the current bookmaker odds); the notebook builds the feature vector, runs it through both models, and clearly reports the predicted winner.

In [ ]:
def get_future_football_features(home_team, away_team, book_odds=None):
    """
    Build a feature vector for a hypothetical future match by averaging each
    team's historical home/away performance from football_df.

    book_odds: optional dict like {'B365H': 2.10, 'B365D': 3.40, 'B365A': 3.20}.
    If omitted, the dataset's average odds are used instead.
    """
    home_hist = football_df[football_df['HomeTeam'] == home_team]
    away_hist = football_df[football_df['AwayTeam'] == away_team]

    if home_hist.empty:
        raise ValueError(f"No historical HOME matches found for '{home_team}'. "
                          f"Available teams: {sorted(football_df['HomeTeam'].unique())}")
    if away_hist.empty:
        raise ValueError(f"No historical AWAY matches found for '{away_team}'. "
                          f"Available teams: {sorted(football_df['AwayTeam'].unique())}")

    feat = {
        'HS':  home_hist['HS'].mean(),  'AS':  away_hist['AS'].mean(),
        'HST': home_hist['HST'].mean(), 'AST': away_hist['AST'].mean(),
        'HF':  home_hist['HF'].mean(),  'AF':  away_hist['AF'].mean(),
        'HC':  home_hist['HC'].mean(),  'AC':  away_hist['AC'].mean(),
        'HY':  home_hist['HY'].mean(),  'AY':  away_hist['AY'].mean(),
        'HR':  home_hist['HR'].mean(),  'AR':  away_hist['AR'].mean(),
    }

    if book_odds is None:
        feat['B365H'] = football_df['B365H'].mean()
        feat['B365D'] = football_df['B365D'].mean()
        feat['B365A'] = football_df['B365A'].mean()
    else:
        feat.update(book_odds)

    return pd.DataFrame([feat])[football_features]


def predict_future_football_match(home_team, away_team, book_odds=None):
    """Predict the winner of a future football match and display results clearly."""
    X_future = get_future_football_features(home_team, away_team, book_odds)
    X_future_s = scaler_fb.transform(X_future)

    rf_probs = rf_fb.predict_proba(X_future)[0]
    ann_probs = ann_fb.predict(X_future_s, verbose=0)[0]

    classes = le_fb.classes_  # ['A','D','H'] (alphabetical)
    label_map = {'H': f'{home_team} (Home Win)',
                 'D': 'Draw',
                 'A': f'{away_team} (Away Win)'}

    results = pd.DataFrame({
        'Outcome': [label_map[c] for c in classes],
        'Random Forest Prob.': rf_probs,
        'ANN Prob.': ann_probs
    })

    rf_winner = label_map[classes[np.argmax(rf_probs)]]
    ann_winner = label_map[classes[np.argmax(ann_probs)]]

    print(f"Predicted outcome for: {home_team} (Home) vs {away_team} (Away)")
    print("=" * 65)
    display(results)

    results.set_index('Outcome')[['Random Forest Prob.', 'ANN Prob.']].plot(
        kind='bar', figsize=(8, 5), colormap='coolwarm')
    plt.title(f"Predicted Win Probabilities: {home_team} vs {away_team}")
    plt.ylabel("Probability")
    plt.ylim(0, 1)
    plt.xticks(rotation=15)
    plt.tight_layout()
    plt.show()

    print(f"\n🏆 Random Forest predicts: {rf_winner}")
    print(f"🏆 ANN predicts: {ann_winner}")

    return results

In [ ]:
# Example: pick two teams from the dataset and predict their next meeting.
# Replace these with any real upcoming fixture's team names.
HOME_TEAM_FUTURE = football_df['HomeTeam'].unique()[0]
AWAY_TEAM_FUTURE = football_df['AwayTeam'].unique()[1]

print(f"Example future fixture: {HOME_TEAM_FUTURE} vs {AWAY_TEAM_FUTURE}\n")

# You can also lock in known pre-match bookmaker odds instead of dataset averages, e.g.:
# _ = predict_future_football_match(HOME_TEAM_FUTURE, AWAY_TEAM_FUTURE,
#                                    book_odds={'B365H': 2.10, 'B365D': 3.40, 'B365A': 3.20})

_ = predict_future_football_match(HOME_TEAM_FUTURE, AWAY_TEAM_FUTURE)

---
# PART 2 — Cricket Match Outcome Prediction

The `deliveries.csv` file contains ball-by-ball data (IPL) without a pre-computed match winner column, so we first **aggregate the ball-by-ball data to match level** and **derive the winner** by comparing the two innings' totals. We then predict the outcome using the first-innings performance (the information available once the team batting first has finished, i.e. before/while the chase is underway) — a realistic "who wins the chase" prediction task.

### 2.1 Load and Aggregate Ball-by-Ball Data

In [ ]:
cricket_raw = pd.read_csv(CRICKET_FILE)
print("Shape:", cricket_raw.shape)
cricket_raw.head()

In [ ]:
# Keep only the two main innings (exclude any super-over innings, coded 3/4)
cricket_raw = cricket_raw[cricket_raw['inning'].isin([1, 2])].copy()

def aggregate_innings(group):
    runs = group['total_runs'].sum()
    wickets = group['is_wicket'].sum()
    extras = group['extra_runs'].sum()
    balls = len(group)
    fours = (group['batsman_runs'] == 4).sum()
    sixes = (group['batsman_runs'] == 6).sum()
    overs = balls / 6 if balls > 0 else 0
    run_rate = runs / overs if overs > 0 else 0
    return pd.Series({
        'team': group['batting_team'].iloc[0],
        'opponent': group['bowling_team'].iloc[0],
        'runs': runs, 'wickets': wickets, 'extras': extras,
        'balls': balls, 'fours': fours, 'sixes': sixes, 'run_rate': run_rate
    })

innings_df = (cricket_raw.groupby(['match_id', 'inning'])
              .apply(aggregate_innings)
              .reset_index())
print("Innings-level rows:", innings_df.shape)
innings_df.head()

### 2.2 Build Match-Level Dataset and Derive the Winner

In [ ]:
records = []
for match_id, g in innings_df.groupby('match_id'):
    g = g.set_index('inning')
    if 1 not in g.index or 2 not in g.index:
        continue  # skip incomplete matches (e.g. abandoned)
    inn1, inn2 = g.loc[1], g.loc[2]

    # Team batting first (team1) wins if it defends its total,
    # i.e. the chasing team (team2) fails to out-score it.
    team1_won = 1 if inn1['runs'] > inn2['runs'] else 0

    records.append({
        'match_id': match_id,
        'team1': inn1['team'], 'team2': inn2['team'],
        'team1_runs': inn1['runs'], 'team1_wickets': inn1['wickets'],
        'team1_extras': inn1['extras'], 'team1_fours': inn1['fours'],
        'team1_sixes': inn1['sixes'], 'team1_run_rate': inn1['run_rate'],
        'team2_extras': inn2['extras'],
        'team1_won': team1_won
    })

cricket_match_df = pd.DataFrame(records)
print("Total matches usable:", cricket_match_df.shape[0])
print("\nOutcome distribution (1 = team batting first wins):")
print(cricket_match_df['team1_won'].value_counts())

sns.countplot(x='team1_won', data=cricket_match_df, palette='magma')
plt.title("Distribution of Match Outcomes (Cricket)")
plt.xlabel("1 = Team batting first won, 0 = Chasing team won")
plt.show()

cricket_match_df.head()

### 2.3 Feature Engineering and Train/Test Split (Cricket)

In [ ]:
le_team1 = LabelEncoder()
le_team2 = LabelEncoder()
cricket_match_df['team1_enc'] = le_team1.fit_transform(cricket_match_df['team1'])
cricket_match_df['team2_enc'] = le_team2.fit_transform(cricket_match_df['team2'])

cricket_features = ['team1_enc', 'team2_enc', 'team1_runs', 'team1_wickets',
                     'team1_extras', 'team1_fours', 'team1_sixes',
                     'team1_run_rate', 'team2_extras']

X_ck = cricket_match_df[cricket_features].copy()
y_ck = cricket_match_df['team1_won'].values

X_ck_train, X_ck_test, y_ck_train, y_ck_test = train_test_split(
    X_ck, y_ck, test_size=0.2, random_state=SEED, stratify=y_ck
)

scaler_ck = StandardScaler()
X_ck_train_s = scaler_ck.fit_transform(X_ck_train)
X_ck_test_s = scaler_ck.transform(X_ck_test)

print("Train size:", X_ck_train.shape, " Test size:", X_ck_test.shape)

### 2.4 Random Forest Model (Cricket)

In [ ]:
rf_ck = RandomForestClassifier(n_estimators=300, max_depth=None, random_state=SEED)
rf_ck.fit(X_ck_train, y_ck_train)

rf_ck_pred = rf_ck.predict(X_ck_test)

rf_ck_acc = accuracy_score(y_ck_test, rf_ck_pred)
rf_ck_prec = precision_score(y_ck_test, rf_ck_pred, average='macro', zero_division=0)
rf_ck_rec = recall_score(y_ck_test, rf_ck_pred, average='macro', zero_division=0)
rf_ck_f1 = f1_score(y_ck_test, rf_ck_pred, average='macro', zero_division=0)

print("Random Forest — Cricket")
print(f"Accuracy : {rf_ck_acc:.4f}")
print(f"Precision: {rf_ck_prec:.4f}")
print(f"Recall   : {rf_ck_rec:.4f}")
print(f"F1-score : {rf_ck_f1:.4f}")
print("\n", classification_report(y_ck_test, rf_ck_pred, zero_division=0))

In [ ]:
cm_ck = confusion_matrix(y_ck_test, rf_ck_pred)
sns.heatmap(cm_ck, annot=True, fmt='d', cmap='Oranges',
            xticklabels=['Chaser won','Team1 won'], yticklabels=['Chaser won','Team1 won'])
plt.title("Random Forest Confusion Matrix — Cricket")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

importances_ck = pd.Series(rf_ck.feature_importances_, index=cricket_features).sort_values(ascending=True)
importances_ck.plot(kind='barh', color='darkorange')
plt.title("Random Forest Feature Importance — Cricket")
plt.xlabel("Importance")
plt.show()

### 2.5 Artificial Neural Network (ANN) Model (Cricket)

In [ ]:
def build_binary_ann(input_dim):
    model = keras.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(32, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(16, activation='relu'),
        layers.Dropout(0.2),
        layers.Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

ann_ck = build_binary_ann(X_ck_train_s.shape[1])
ann_ck.summary()

In [ ]:
early_stop_ck = keras.callbacks.EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)

history_ck = ann_ck.fit(
    X_ck_train_s, y_ck_train,
    validation_split=0.15,
    epochs=150,
    batch_size=16,
    callbacks=[early_stop_ck],
    verbose=0
)

print("Training complete. Epochs run:", len(history_ck.history['loss']))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(history_ck.history['loss'], label='Train Loss')
ax[0].plot(history_ck.history['val_loss'], label='Val Loss')
ax[0].set_title("ANN Loss — Cricket")
ax[0].set_xlabel("Epoch"); ax[0].legend()

ax[1].plot(history_ck.history['accuracy'], label='Train Acc')
ax[1].plot(history_ck.history['val_accuracy'], label='Val Acc')
ax[1].set_title("ANN Accuracy — Cricket")
ax[1].set_xlabel("Epoch"); ax[1].legend()
plt.tight_layout()
plt.show()

In [ ]:
ann_ck_probs = ann_ck.predict(X_ck_test_s, verbose=0).ravel()
ann_ck_pred = (ann_ck_probs >= 0.5).astype(int)

ann_ck_acc = accuracy_score(y_ck_test, ann_ck_pred)
ann_ck_prec = precision_score(y_ck_test, ann_ck_pred, average='macro', zero_division=0)
ann_ck_rec = recall_score(y_ck_test, ann_ck_pred, average='macro', zero_division=0)
ann_ck_f1 = f1_score(y_ck_test, ann_ck_pred, average='macro', zero_division=0)

print("ANN — Cricket")
print(f"Accuracy : {ann_ck_acc:.4f}")
print(f"Precision: {ann_ck_prec:.4f}")
print(f"Recall   : {ann_ck_rec:.4f}")
print(f"F1-score : {ann_ck_f1:.4f}")
print("\n", classification_report(y_ck_test, ann_ck_pred, zero_division=0))

### 2.6 Cricket: Random Forest vs ANN Comparison

In [ ]:
cricket_results = pd.DataFrame({
    'Model': ['Random Forest', 'ANN'],
    'Accuracy': [rf_ck_acc, ann_ck_acc],
    'Precision': [rf_ck_prec, ann_ck_prec],
    'Recall': [rf_ck_rec, ann_ck_rec],
    'F1-score': [rf_ck_f1, ann_ck_f1]
})
display(cricket_results)

cricket_results.set_index('Model').plot(kind='bar', figsize=(8,5), colormap='Set3')
plt.title("Cricket: Random Forest vs ANN")
plt.ylabel("Score")
plt.ylim(0, 1)
plt.xticks(rotation=0)
plt.legend(loc='lower right')
plt.show()

### 2.7 Predict a Future Cricket Match

Using the same approach, we can predict the outcome of an upcoming match between two IPL teams. We estimate `team1`'s (the side batting first) expected first-innings performance from its historical average **when batting first**, and `team2`'s expected extras conceded from its historical average **when bowling second/chasing**.

Supply the two team names (which one bats first matters, since the model was trained on that framing) and the notebook reports both models' predicted winner side-by-side.

In [ ]:
def get_future_cricket_features(team1, team2):
    """
    team1 = team assumed to bat first, team2 = team assumed to chase.
    Estimates the feature vector from each team's historical role-specific averages.
    """
    t1_hist = cricket_match_df[cricket_match_df['team1'] == team1]
    t2_hist = cricket_match_df[cricket_match_df['team2'] == team2]

    if t1_hist.empty:
        raise ValueError(f"No historical 'batting first' matches for '{team1}'. "
                          f"Available teams: {sorted(cricket_match_df['team1'].unique())}")
    if t2_hist.empty:
        raise ValueError(f"No historical 'chasing' matches for '{team2}'. "
                          f"Available teams: {sorted(cricket_match_df['team2'].unique())}")

    def safe_encode(le, value, fallback_series):
        if value in le.classes_:
            return le.transform([value])[0]
        # Unseen team name: fall back to the most frequent encoded value
        return le.transform([fallback_series.mode()[0]])[0]

    feat = {
        'team1_enc': safe_encode(le_team1, team1, cricket_match_df['team1']),
        'team2_enc': safe_encode(le_team2, team2, cricket_match_df['team2']),
        'team1_runs': t1_hist['team1_runs'].mean(),
        'team1_wickets': t1_hist['team1_wickets'].mean(),
        'team1_extras': t1_hist['team1_extras'].mean(),
        'team1_fours': t1_hist['team1_fours'].mean(),
        'team1_sixes': t1_hist['team1_sixes'].mean(),
        'team1_run_rate': t1_hist['team1_run_rate'].mean(),
        'team2_extras': t2_hist['team2_extras'].mean(),
    }
    return pd.DataFrame([feat])[cricket_features]


def predict_future_cricket_match(team1, team2):
    """Predict the winner of a future cricket match and display results clearly."""
    X_future = get_future_cricket_features(team1, team2)
    X_future_s = scaler_ck.transform(X_future)

    rf_prob_team1 = rf_ck.predict_proba(X_future)[0][1]
    ann_prob_team1 = float(ann_ck.predict(X_future_s, verbose=0)[0][0])

    results = pd.DataFrame({
        'Model': ['Random Forest', 'ANN'],
        f'{team1} Win Prob.': [rf_prob_team1, ann_prob_team1],
        f'{team2} Win Prob.': [1 - rf_prob_team1, 1 - ann_prob_team1]
    })

    rf_winner = team1 if rf_prob_team1 >= 0.5 else team2
    ann_winner = team1 if ann_prob_team1 >= 0.5 else team2

    print(f"Predicted outcome for: {team1} (batting first) vs {team2} (chasing)")
    print("=" * 65)
    display(results)

    results.set_index('Model').plot(kind='bar', figsize=(8, 5), colormap='cool')
    plt.title(f"Predicted Win Probabilities: {team1} vs {team2}")
    plt.ylabel("Probability")
    plt.ylim(0, 1)
    plt.xticks(rotation=0)
    plt.axhline(0.5, color='gray', linestyle='--', linewidth=1)
    plt.tight_layout()
    plt.show()

    print(f"\n🏆 Random Forest predicts: {rf_winner} wins")
    print(f"🏆 ANN predicts: {ann_winner} wins")

    return results

In [ ]:
# Example: pick two teams from the dataset and predict their next meeting.
# Replace these with any real upcoming fixture's team names.
TEAM1_FUTURE = cricket_match_df['team1'].unique()[0]
TEAM2_FUTURE = cricket_match_df['team2'].unique()[1]

print(f"Example future fixture: {TEAM1_FUTURE} (bat first) vs {TEAM2_FUTURE} (chase)\n")

_ = predict_future_cricket_match(TEAM1_FUTURE, TEAM2_FUTURE)

---
# 3. Overall Comparison — Cricket vs Football

Finally, we bring together the results from both sports to see how each model performs across domains.

In [ ]:
overall = pd.DataFrame({
    'Sport': ['Football', 'Football', 'Cricket', 'Cricket'],
    'Model': ['Random Forest', 'ANN', 'Random Forest', 'ANN'],
    'Accuracy': [rf_fb_acc, ann_fb_acc, rf_ck_acc, ann_ck_acc],
    'Precision': [rf_fb_prec, ann_fb_prec, rf_ck_prec, ann_ck_prec],
    'Recall': [rf_fb_rec, ann_fb_rec, rf_ck_rec, ann_ck_rec],
    'F1-score': [rf_fb_f1, ann_fb_f1, rf_ck_f1, ann_ck_f1]
})
display(overall)

fig, ax = plt.subplots(figsize=(9, 5))
x = np.arange(len(overall))
ax.bar(x, overall['Accuracy'], color=['#4C72B0','#55A868','#C44E52','#8172B2'])
ax.set_xticks(x)
ax.set_xticklabels(overall['Sport'] + " - " + overall['Model'], rotation=20, ha='right')
ax.set_ylabel("Accuracy")
ax.set_ylim(0, 1)
ax.set_title("Model Accuracy Comparison: Cricket vs Football")
plt.tight_layout()
plt.show()

## 4. Conclusion

- **Random Forest** provides a strong, interpretable baseline for both sports and highlights the most influential features (e.g. shots on target and bookmaker odds for football; first-innings run rate and wickets for cricket).
- The **ANN** learns non-linear feature interactions and produces comparable performance, illustrating the trade-off between interpretability (Random Forest) and pattern-recognition capacity (ANN) discussed in the project abstract.
- This hybrid evaluation framework (Random Forest + ANN, compared with Accuracy/Precision/Recall/F1) can be extended with richer features (player-level form, venue effects, head-to-head history) to further improve predictive reliability for real-world sports analytics applications.